
 ### Метод наименьших квадратов (MNK). Аппроксимация данных.

 mms, 6.11.23/25, 15.10.26

 Метод наименьших квадратов используется для решения задачи **аппроксимации** зашумленных данных. Хотим найти функцию, которая наилучшим образом описывает данные. Отличие от задачи интерполяции состоит в том, что здесь от искомой функции не требуется точного попадания в узлы.  

Пусть имееется набор данных $(x_i, y_i),  i=\overline{1,m}$. Требуется найти функцию  $F(x)$, для которой отклонение от данных в узлах имеет малую погрешность $\eta$:
$$
y_i=F(x_i)-\eta_i  \tag{1}
$$

Будем искать $F(x)$ в виде линейной комбинации каких-то заранее выбранных базисных функций $f_j$:
$$
F(x)=\sum_{j=1}^n c_j f_j(x)   \tag{2}
$$
Цель - найти коэффициенты $c_j$.

Пусть $A$ - $m\times n$ матрица значений базисных функций в узлах:
$$
A =
\left(
\begin{array}{cc}
f_1(x_1) & f_2(x_1) & \dots & f_n(x_1) \\
f_1(x_2) & f_2(x_2) & \dots & f_n(x_2) \\
\dots \\
f_1(x_m) & f_2(x_m) & \dots & f_n(x_m) \\
\end{array}
\right)
\qquad
$$
Тогда (2) можно записать в виде:
$$
Ac=F,
$$
где $с$ - вектор из $n$ искомых коэфициентов, а $F$ - вектор из $m$ значений искомой функции в узлах.

Тогда (1):
$$
\eta=Ac-y,
$$
где $\eta$ - вектор *невязки*. Нужно решить это уравнение так, чтобы этот вектор невязки был как можно меньше. Выбор меры близости к решению - критерий MНK (т.е. минимизация суммы квадратов отклонений искомой кривой от заданных точек):
$$
\min{\|\eta\|_2^2}=\|Ac-y\|_2^2=\sum_{i=1}^m(\sum_{j=1}^n a_{ij}c_j-y_i)^2
$$
Минимум  - занулим производные по всем переменным, получим систему из $n$ уравнений:
$$
\frac{\partial}{\partial c_k}\|\eta\|_2^2= \sum_{i=1}^m(2\sum_{j=1}^n a_{ij}c_j-y_i)a_{ik} = 0,
$$
или
$$
(Ac-y)^{\top}A=0 => A^{\top}(Ac-y)=0 => A^{\top}Ac=A^{\top}y =>
$$

$$
c=(A^{\top}A)^{-1}A^{\top}y=A^+ y
$$

Таким образом, мы выразили искомый вектор коэффициентов через матрицу значений базисных функций в узлах и вектор у.  
Заметим, что матрица $A$ не квадратная, а $(A^{\top}A)$ квадратная с размерностью числа базисных функций $n$. Важна обратимость этой матрицы, чего может не быть при неудачном выборе базисного набора. Здесь мы вводим псевдообранную матрицу $A^+$, которая практически всегда вычислима. И далее в прямом расчете (п.1) будем использовать именно ее.  



**1. Прямой способ - через псевдообратную матрицу**

In [13]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
x = np.array([-1, 1, 2, 3, 5])
y = np.array([2, 1, 1, 0, 3])

In [ ]:
# Создаем матрицу A для базисных функций (1, x, x^2)
A=np.zeros((5, 3))
A[:,0]=1
A[:,1]=x
A[:,2]=x*x
A

In [ ]:
# то же самое можно так (это короче), cм.help(np.vstack)
A = np.vstack([np.ones(len(x)),x,x*x]).T
A

In [ ]:
# Решение: находим коэф-ты для базисных функций (1, x, x^2)
c=np.dot(np.linalg.pinv(A),y.T)  # можно не указывать транспонирование вектора y.T, будет сделано автоматически
print(c)

In [ ]:
def F(c,x):
    return c[0]+c[1]*x+c[2]*x*x

In [ ]:
#Cравним с интерполяцией полиномом 2-й степени (polifit).
#Поскольку число N=5, а степень полинома 2<N-1, то в этом случае polifit тоже использует алгоритм MNK.
p1 = np.poly1d(np.polyfit(x, y, 2))
print(np.polyfit(x, y, 2))        # !коэффициенты в обратном порядке
print(p1)

In [ ]:
# Сравним ошибки (первую и вторую нормы отклонений по всем точкам)
print(f' norm1(p1)={np.linalg.norm(p1(x)-y,ord=1)}\n norm1(F) ={np.linalg.norm(F(c,x)-y,ord=1)}')
print(f' norm2(p1)={np.linalg.norm(p1(x)-y)}\n norm2(F) ={np.linalg.norm(F(c,x)-y)}')

In [ ]:
# Графики
xn=np.linspace(-2,6,100)
plt.plot(x,y,'o', label='Data')
plt.plot(xn,F(c,xn), 'g--', lw=3, label='LSQ')
plt.plot(xn, p1(xn), 'r', label='poly1d')
plt.legend()
plt.show()
plt.plot(xn, abs(F(c,xn)-p1(xn)))
plt.title("Абсолютное отклонение расчета прямым MНК \n и интерполяцией полиномом 2-й степени")
plt.show()

In [ ]:
# Задание 1. Даны точки (1,1) (2,1) (3,3) (4,8). Построить аппроксимацию вида F(x)=c1+c2*x*log(x)+c3*exp(x).
# Напечатать коэффициенты, среднеквадратичную ошибку(2-норма) и построить график полученной F(x).


**2. Использование встроенных функций**  


In [ ]:
# 2. Использование встроенных функций линейной аппроксимации (lstsq: Least Squares)
# 2.1  np.linalg.lstsq
# 2.2  scipy.linalg.lstsq

In [ ]:
### 2.1  np.linalg.lstsq
# Приближение линейной функцией с помощью numpy.linalg.lstsq

import numpy as np
x = np.array([0, 1, 2, 3])
y = np.array([-1, 0.2, 0.9, 2.1])

#Построим матрицу A для базисных функций (1, x):
A = np.vstack([x, np.ones(len(x))]).T
print(A)

In [ ]:
#Используем lstsq для поиска коэффициентов
#help(np.linalg.lstsq):
# - [0] — берет только первый элемент из возвращаемого списка (сам вектор x),
#отбрасывая остатки (residuals), ранг матрицы (rank) и сингулярные числа (s).

#c = np.linalg.lstsq(A, y, rcond=None)[0]
c = np.linalg.lstsq(A, y)[0]
print (c)

In [ ]:
#для сравнения найдем коэффициенты по старому варианту (должны совпадать)
c_old=np.dot(np.linalg.pinv(A),y)
print(c_old)

In [ ]:
#График
plt.plot(x, y, 'o',markersize=5)
plt.plot(x, c[0]*x + c[1], 'r')
plt.show()

In [9]:
### 2.2  scipy.linalg.lstsq
# Пример аппроксимации параболой с помощью функции scipy.linalg.lstsq по зашумленным данным
# Данные должны быть получены по измерениям, но в этом примере,сделаем эти данные сами.
# Просто возьмем квадратичную функцию и добавим случайные отклонения в координаты х и у.

from numpy import *
from numpy.random import *

x=linspace(-5,5,15)            # сгенерируем 15 точек
y=x**2+(rand(15)-0.5)
x+=rand(15)-0.4
#x.tofile('x_data.txt', '\n')   # можно сохранить данные в файл
#y.tofile('y_data.txt', '\n')
#x,y
#plt.plot(x, y, 'o',markersize=5)   # и посмотреть на экспериментальные точки
#plt.show()

In [ ]:
import scipy.linalg as sl

# читаем данные из файлов
# x1=fromfile('x_data.txt',float,sep='\n')
# y1=fromfile('y_data.txt',float,sep='\n')
# print(x1,y1)

# задаем матрицу А для [x**2, x, 1]
A=np.vstack((x**2,x,ones(15))).T

# находим коэффициенты при компонентах вектора
s=sl.lstsq(A,y)[0]
print(s)

In [ ]:
#тест для сравнения по первому варианту
c_old=np.dot(np.linalg.pinv(A),y)
print(c_old)

In [ ]:
# Графики
xn=linspace(-5,5,100)
# рисуем теоретическую кривую x^2 и "экспериментальные" точки
plt.plot(xn, xn**2,'--',label='Initial')
plt.plot(x,y,'D',label='Data')

# и нашу квадратичную аппроксимацию с коэффициентами s[0], s[1], s[2]
plt.plot(xn,s[0]*xn**2+s[1]*xn+s[2],'-',label='Lstsq fit')
plt.legend()
plt.grid()
plt.show()



**3.1 Функция scipy.optimize.curve_fit - возможность не только линейной аппроксимации**


In [50]:
### 3.  scipy.optimize.curve_fit(func, x, y)
# Может подгонять коэффициенты _произвольной_ функции, не обязательно линейной по коэффициентам.
# Пример с функцией вида f(x,b) = b0 + b1*exp(-b2*x^2)

from scipy.optimize import curve_fit

# Генерация данных на функции с b = (0.25, 0.75, 0.5) и добавление шума

b = (0.25, 0.75, 0.5)

def F3(x, b0, b1, b2):
    return b0 + b1*np.exp(-b2*x**2)


xdata = np.linspace(0, 5, 50)                   # создаем данные по х
y = F3(xdata, *b)                               # создаем y без шума

np.random.seed(301)                             # fix seed дла воспроизводимости
ydata = y + 0.2 * np.random.randn(len(xdata))   # добавляем шум

In [ ]:
# С помощью curve_fit находим коэффифицетны b_opt и ковариационную матрицу b_cov
# help(curve_fit)
b_opt,b_cov = curve_fit(F3, xdata, ydata)
print("params b:", b_opt)

# ковариационная матрица (см. далее., не обязательно, но весьма полезно)
#print(b_cov)
print("cond:", np.linalg.cond(b_cov))    # число обусловленности ковариационной матрицы является
                                         # оценкой качества параметризации. Не должно быть очень большим.

In [ ]:
# Можно посмотреть норму невязки (пример расчета вручную, <=> np.linalg.norm)
residuals = ydata - F3(xdata,*b_opt)
fres = sum(residuals**2)                       # сумма квадратов отклонений по всем точкам,
                                               # по нему вычисляется норма вектора невязки: sqrt(fres)
print(fres)
print(np.sqrt(fres),"<=>", np.linalg.norm(residuals))

In [ ]:
#fig, ax = plt.subplots()
plt.plot(xdata, ydata, 'o',label="Data")
plt.plot(xdata, y, 'r', lw=2, label="Theor")
plt.plot(xdata, F3(xdata, *b_opt), 'g', label="Fit")
plt.legend()
plt.show()

**3.2 Ковариационная матрица**

In [ ]:
#Для получения надежных результатов модельную функцию не следует перепараметризировать;
#избыточные параметры могут привести к ненадежным ковариационным матрицам и, нередко,
#к ухудшению качества подгонки. Для быстрой проверки на избыточность параметров модели можно
#вычислить число обусловленности ковариационной матрицы, которое не должно быть очень большим:

print(b_cov)                    # ков. матрица
print(np.linalg.cond(b_cov))    # число обусловленности

In [ ]:
# Пример для иллюстрации излишней параметризации
# Добавим к рассмотренной выше функции еще один параметр(b3) : f(x,b) = b0 + b3*b1*exp(-b2*x^2)

b = (0.25, 0.75, 0.5, 1)

def F3(x, b0, b1, b2, b3):
    return b0 + b3*b1*np.exp(-b2*x**2)

xdata = np.linspace(0, 5, 50)                   # создаем данные по х
y = F3(xdata, *b)                               # создаем y без шума
ydata = y + 0.2 * np.random.randn(len(xdata))
b_opt,b_cov = curve_fit(F3, xdata, ydata)

print("b_opt:",b_opt)         # коэф-ты менее похожи на оригинал (0.25, 0.75, 0.5);
#print(b_cov)

print("cond:", np.linalg.cond(b_cov))  # число обусловленности очень велико;

# Диагональные элементы ков. матрицы, связаны с неопределенностью соответствия:
print("diag:", np.diag(b_cov))     # второй и последний очень большие, что говорит о
                                   # неоднозначности параметров b1 и b3, в модели нужен
                                   # только один из них.

**Задания**

In [ ]:
#Задание 2. Аппроксимируйте функцией H(x)=c(1)+c(2)*x*sin(x)+c(3)*cos(4*x)
#cледующий набор измерений
#xdata= [0.5, 1.5, 2.5, 3.5, 4.5, 5.5, 6.5,7.5];
#ydata=[1.2, 2.7, 0.8, -1.1, -2.0, -3.0, 1.2, 6.0]
#Выведите графики и найденные коэффициенты с(i).
#Вычислите норму вектора невязки R(сумма квадратов отклонений по всем точкам) и
#среднеквадратическое отклонение sqrt(R/m), где m-число точек.

In [ ]:
# Задание 3.
# Есть данные:
# X=[ 0.  1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12.]
# Y=[0, 2.18, 2.82, 1.78, 0.48, 0.58, 2.44, 4.81, 5.98,5.32,3.91,3.5, 4.93]
# Выберите подходящий базис для линейной комбинации и постройте аппроксимационную функцию.

In [ ]:
# Задание 4. Напишите функцию, которая принимает данные (векторы x и y произвольной размерности)
# и возвращает параметры a и b линейной аппрокимирующей функции F(x)=ax+b. Не используйте для этого
# этого pinv, а выведите явные формулы для решения системы двух линейных уравнений на a и b.
# Проверьте работу своей реализации, напр, на примере 2.1.


In [ ]:
# Задание 5**. Выражение z=a*x^2 + b*x*y +c*y^2 + d*x + e*y + f определяет квадратичную форму двух
# переменных (x,y). Множество точек (x,у), таких, что z=0 называется коническим сечением, которое
# может быть эллипсом, параболой или гиперболой в зависимости от значения b^2 - 4*a*c.
# 1)Постройте график конического сечения (использйте np.meshgrid(x, y) и plt.contour). Задавая подходящие
# коэффициенты, постройте эллипс, параболу и гиперболу.
# 2) Известно, что согласно законам Кеплера планетарные орбиты являются коническими сечениями. Пусть
# даны астрономические наблюдения координат движения небесного тела в плоскости (x,y):
#   x=[1.02,0.95,0.87,0.77,0.67,0.56,0.44,0.30,0.16,0.01]
#   y=[0.39,0.32,0.27,0.22,0.18,0.15,0.13,0.12,0.13,0.15]
# Определите методом наименьших квадратов коэф-ты квадратичной формы и постройте график орбиты.
# 3) Используя генератор случайных чисел, добавьте к каждому измерению шум в интервале [-0.005,0.005].
# Вычислите новые коэф-ты и постройте на одном графике старую орбиту и возмущенную новую орбиту.
# Обясните полученный результат.